# Notebook 01 · Un entorno de refuerzo desde cero

**Introducción al Deep Learning — Módulo 4, Unidad 2: Aprendizaje por refuerzo (Parte I)**

En el aprendizaje supervisado le dábamos a la red un conjunto de datos con las respuestas correctas.
En el **aprendizaje por refuerzo** no hay respuestas: hay un **agente** que actúa y un **entorno** que le
devuelve un **nuevo estado** y una **recompensa**.

En este notebook vamos a construir ese bucle con nuestras propias manos, sin librerías de RL, para que
quede claro qué es cada pieza:

| Elemento | En nuestro ejemplo |
|---|---|
| **Entorno** | Una rejilla 4×4 |
| **Estado** | La casilla donde está el agente |
| **Acción** | Moverse arriba, abajo, izquierda o derecha |
| **Refuerzo** | +10 al llegar a la meta, −10 en una trampa, −1 por cada paso |
| **Agente** | Quien elige la acción (primero al azar, luego con una política) |

> 💡 Solo usamos NumPy y Matplotlib: todo se ejecuta en segundos.

## 1. El entorno: una rejilla 4×4

El agente empieza arriba a la izquierda (casilla `0,0`) y quiere llegar a la meta (abajo a la derecha).
Hay dos **trampas** que hay que evitar.

El `-1` por cada paso es importante: obliga al agente a buscar el camino **corto**, no cualquier camino.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

FILAS, COLS = 4, 4
META = (3, 3)
TRAMPAS = [(1, 1), (2, 3)]

ACCIONES = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # arriba, abajo, izq, der
NOMBRES  = {0: "arriba", 1: "abajo", 2: "izquierda", 3: "derecha"}

def es_terminal(estado):
    return estado == META or estado in TRAMPAS

def paso(estado, accion):
    '''Recibe (estado, accion) y devuelve (nuevo_estado, refuerzo, fin).

    Esta funcion ES el entorno: es lo unico que el agente puede 'preguntar'.'''
    df, dc = ACCIONES[accion]
    f = min(max(estado[0] + df, 0), FILAS - 1)     # si choca con el borde, se queda
    c = min(max(estado[1] + dc, 0), COLS - 1)
    nuevo = (f, c)
    if nuevo == META:
        return nuevo, 10.0, True
    if nuevo in TRAMPAS:
        return nuevo, -10.0, True
    return nuevo, -1.0, False

print("Entorno listo. Prueba de un paso desde (0,0) hacia la derecha:")
print(paso((0, 0), 3))

### Cómo se ve la rejilla

In [ ]:
def dibujar(estado=None, titulo=""):
    fig, ax = plt.subplots(figsize=(3.6, 3.6))
    rejilla = np.zeros((FILAS, COLS))
    for t in TRAMPAS:
        rejilla[t] = -1
    rejilla[META] = 1
    ax.imshow(rejilla, cmap="RdYlGn", vmin=-1.5, vmax=1.5)
    for f in range(FILAS):
        for c in range(COLS):
            txt = "META" if (f, c) == META else ("X" if (f, c) in TRAMPAS else "")
            ax.text(c, f, txt, ha="center", va="center", fontsize=9)
    if estado is not None:
        ax.plot(estado[1], estado[0], "o", color="navy", markersize=18)
    ax.set_xticks(range(COLS)); ax.set_yticks(range(FILAS))
    ax.set_title(titulo); plt.show()

dibujar((0, 0), "Estado inicial (agente en azul)")

## 2. Un agente que actúa al azar

El agente más simple posible: elige una de las cuatro acciones **sin mirar el estado**.
Vamos a dejarle jugar un **episodio** (desde el inicio hasta que termina) y registrar la recompensa acumulada.

In [ ]:
def agente_aleatorio(estado):
    return np.random.randint(4)

def episodio(politica, max_pasos=50, verboso=False):
    estado, total, camino = (0, 0), 0.0, [(0, 0)]
    for t in range(max_pasos):
        a = politica(estado)
        nuevo, r, fin = paso(estado, a)
        total += r
        camino.append(nuevo)
        if verboso:
            print(f"  paso {t+1}: {estado} --{NOMBRES[a]}--> {nuevo} | refuerzo {r:+.0f}")
        estado = nuevo
        if fin:
            break
    return total, camino

print("Un episodio del agente aleatorio:")
total, camino = episodio(agente_aleatorio, verboso=True)
print(f"\nRecompensa acumulada: {total:+.0f}")

Fíjate en lo importante: el agente **no sabía nada** del entorno. Solo actuó y recibió números.
Ejecuta la celda varias veces: unas veces acaba en la meta y otras en una trampa.

### ¿Y de media?

In [ ]:
resultados_azar = [episodio(agente_aleatorio)[0] for _ in range(500)]
print(f"Recompensa media (agente aleatorio): {np.mean(resultados_azar):+.2f}")
print(f"Mejor episodio: {np.max(resultados_azar):+.0f} | peor: {np.min(resultados_azar):+.0f}")

plt.figure(figsize=(6, 3))
plt.hist(resultados_azar, bins=25, color="#4355FF", edgecolor="white")
plt.title("Recompensa acumulada en 500 episodios (agente aleatorio)")
plt.xlabel("recompensa"); plt.ylabel("episodios"); plt.show()

## 3. Una política escrita a mano

Una **política π** es una función que, dado un estado, dice qué acción tomar. La del agente aleatorio era
una política (mala). Vamos a escribir una razonable: *avanza hacia la meta, evitando trampas*.

Ojo: esta política la escribimos **nosotros** porque conocemos el mapa. El objetivo del RL es que el agente
la **descubra solo** — eso lo veremos en la Parte II con Q-Learning.

In [ ]:
def politica_manual(estado):
    f, c = estado
    candidatas = []
    if f < META[0]: candidatas.append(1)   # bajar
    if c < META[1]: candidatas.append(3)   # ir a la derecha
    # descartamos las que llevan directamente a una trampa
    seguras = [a for a in candidatas
               if (min(max(f + ACCIONES[a][0], 0), FILAS-1),
                   min(max(c + ACCIONES[a][1], 0), COLS-1)) not in TRAMPAS]
    opciones = seguras if seguras else candidatas
    return int(np.random.choice(opciones)) if opciones else 0

total, camino = episodio(politica_manual, verboso=True)
print(f"\nRecompensa acumulada: {total:+.0f}")

In [ ]:
resultados_pol = [episodio(politica_manual)[0] for _ in range(500)]
print(f"Aleatorio : {np.mean(resultados_azar):+.2f}")
print(f"Política  : {np.mean(resultados_pol):+.2f}")

plt.figure(figsize=(6, 3))
plt.hist(resultados_azar, bins=25, alpha=0.6, label="aleatorio", color="#4355FF")
plt.hist(resultados_pol,  bins=25, alpha=0.8, label="política manual", color="#FF647E")
plt.legend(); plt.title("La política importa"); plt.xlabel("recompensa acumulada"); plt.show()

### El camino que recorre la política

In [ ]:
_, camino = episodio(politica_manual)
fig, ax = plt.subplots(figsize=(3.6, 3.6))
rejilla = np.zeros((FILAS, COLS))
for t in TRAMPAS: rejilla[t] = -1
rejilla[META] = 1
ax.imshow(rejilla, cmap="RdYlGn", vmin=-1.5, vmax=1.5)
cs = np.array(camino)
ax.plot(cs[:, 1], cs[:, 0], "-o", color="navy", linewidth=2)
for i, (f, c) in enumerate(camino):
    ax.text(c + 0.22, f - 0.22, str(i), fontsize=8, color="navy")
ax.set_title("Camino seguido (los números son el orden)")
ax.set_xticks(range(COLS)); ax.set_yticks(range(FILAS)); plt.show()

## 4. El factor de descuento γ

Hasta ahora hemos sumado las recompensas tal cual. En RL se suele usar el **retorno descontado**:

$$G = r_1 + \gamma\, r_2 + \gamma^2 r_3 + \dots$$

γ (entre 0 y 1) dice cuánto nos importa el futuro:

- **γ ≈ 0** → el agente es *impaciente*: solo cuenta la recompensa inmediata.
- **γ ≈ 1** → el agente es *paciente*: una recompensa lejana vale casi lo mismo que una inmediata.

In [ ]:
def retorno_descontado(recompensas, gamma):
    return sum((gamma ** t) * r for t, r in enumerate(recompensas))

# un camino largo pero que acaba bien: cuatro pasos de -1 y luego +10
recompensas = [-1, -1, -1, -1, 10]
for g in [0.0, 0.5, 0.9, 0.99, 1.0]:
    print(f"gamma = {g:<5} -> retorno = {retorno_descontado(recompensas, g):+7.3f}")

Con **γ = 0** el retorno es −1: el agente solo ve el primer paso doloroso y ni se entera del premio final.
Con **γ = 0,99** el retorno es positivo: el premio de llegar a la meta compensa el camino.

**Elegir γ es elegir cuánta paciencia tiene el agente.** Es uno de los hiperparámetros clave del RL.

## Experimenta tú

1. Cambia las recompensas: pon `-0.1` por paso en lugar de `-1`. ¿Cambia el comportamiento de la política?
2. Añade una tercera trampa o mueve la meta y vuelve a comparar aleatorio vs. política.
3. Amplía la rejilla a 6×6. ¿Cuánto empeora el agente aleatorio? (pista: mucho — el espacio de estados crece)

## Resumen

- Un problema de RL es un **bucle**: estado → acción → (nuevo estado, refuerzo) → …
- El **entorno** es una función que responde a las acciones; el **agente** solo ve estados y números.
- Una **política π** mapea estados a acciones. Una buena política mejora enormemente la recompensa.
- El **factor de descuento γ** decide cuánto pesa el futuro frente al presente.

➡️ En el **Notebook 02** calcularemos, con la **ecuación de Bellman**, cuánto vale cada casilla.